# 01_mock: Tool-Using Support Agent

Timebox: **55 minutes**  
Language: **Python (Colab)**

This is the highest-priority mock for your interview shape.

## Scenario
Implement an agent loop that uses local tools to inspect orders, evaluate policy, and submit refunds.

## What to implement
1. `validate_tool_call`
2. `execute_tool_call`
3. `run_agent`

## Completion criteria (required)
- Correct tool-use loop (`tool_use` -> execute -> append tool messages -> continue)
- Multiple tool calls in one response
- Error-safe behavior for invalid args + runtime exceptions
- `max_steps` protection against infinite loops

## Time guidance
- 10 min: read scaffold + plan flow
- 35 min: implement required functions
- 10 min: run tests + edge-case cleanup


In [ ]:
import inspect
import json
from copy import deepcopy
from typing import Any, Callable

ORDERS_DB = {
    "u-100": [
        {"order_id": "o-900", "status": "delivered", "days_since_delivery": 3, "amount": 42.5}
    ],
    "u-200": [
        {"order_id": "o-901", "status": "in_transit", "days_since_delivery": 0, "amount": 81.0}
    ],
}
REFUNDS: list[dict[str, Any]] = []


def reset_state() -> None:
    REFUNDS.clear()


def get_orders(user_id: str) -> list[dict[str, Any]]:
    if user_id == "boom":
        raise RuntimeError("orders backend unavailable")
    return deepcopy(ORDERS_DB.get(user_id, []))


def policy_check(order_id: str, reason: str, days_since_delivery: int) -> dict[str, Any]:
    eligible = reason.lower() in {"damaged", "wrong_item"} and days_since_delivery <= 14
    return {
        "order_id": order_id,
        "eligible": eligible,
        "policy_reason": "allowed" if eligible else "outside_policy",
    }


def create_refund(order_id: str, amount: float) -> dict[str, Any]:
    if amount <= 0:
        raise ValueError("refund amount must be positive")
    record = {"order_id": order_id, "amount": amount, "status": "submitted"}
    REFUNDS.append(record)
    return deepcopy(record)


TOOL_REGISTRY: dict[str, Callable[..., Any]] = {
    "get_orders": get_orders,
    "policy_check": policy_check,
    "create_refund": create_refund,
}


class ScriptedModel:
    def __init__(self, responses: list[dict[str, Any]]) -> None:
        self._responses = deepcopy(responses)
        self._index = 0

    def __call__(self, messages: list[dict[str, Any]]) -> dict[str, Any]:
        if self._index >= len(self._responses):
            return {"stop_reason": "end_turn", "output_text": "No scripted response left."}
        response = self._responses[self._index]
        self._index += 1
        return deepcopy(response)


In [ ]:
def validate_tool_call(tool_call: dict[str, Any], tool_registry: dict[str, Callable[..., Any]]) -> str | None:
    """Return an error string if invalid, otherwise None."""
    # TODO:
    # 1) Ensure tool_call has id/name/input.
    # 2) Ensure tool exists in registry.
    # 3) Ensure input is a dict.
    # 4) Ensure all required function args are present.
    raise NotImplementedError


def execute_tool_call(tool_call: dict[str, Any], tool_registry: dict[str, Callable[..., Any]]) -> dict[str, Any]:
    """Return tool message with is_error and JSON content."""
    # TODO:
    # - Call validate_tool_call first.
    # - Execute valid tools with **tool_call["input"].
    # - Catch runtime exceptions and return is_error=True payload.
    # Return shape:
    # {
    #   "role": "tool",
    #   "tool_call_id": "...",
    #   "name": "...",
    #   "is_error": bool,
    #   "content": "json-string"
    # }
    raise NotImplementedError


def run_agent(
    user_prompt: str,
    model: Callable[[list[dict[str, Any]]], dict[str, Any]],
    tool_registry: dict[str, Callable[..., Any]],
    max_steps: int = 6,
) -> dict[str, Any]:
    """Run tool-use loop until end_turn or max_steps exhaustion."""
    # TODO:
    # - Initialize messages with user prompt.
    # - Loop up to max_steps.
    # - On stop_reason == "tool_use", execute all tool calls and append tool messages.
    # - On stop_reason == "end_turn", return {"final_text": ..., "messages": ...}.
    # - Raise RuntimeError("max_steps_exceeded") if no end_turn in time.
    raise NotImplementedError


## Run Tests
Run this final test cell after implementing all TODO sections.


In [ ]:
def _tool_messages(messages: list[dict[str, Any]]) -> list[dict[str, Any]]:
    return [m for m in messages if m.get("role") == "tool"]


def run_exam01_tests() -> None:
    reset_state()

    # 1) Single tool call
    model = ScriptedModel(
        [
            {
                "stop_reason": "tool_use",
                "tool_calls": [{"id": "t1", "name": "get_orders", "input": {"user_id": "u-100"}}],
            },
            {"stop_reason": "end_turn", "output_text": "Order o-900 is delivered."},
        ]
    )
    result = run_agent("Where is my order?", model, TOOL_REGISTRY)
    assert "delivered" in result["final_text"].lower()
    assert len(_tool_messages(result["messages"])) == 1

    # 2) Multiple tools in one model turn
    model = ScriptedModel(
        [
            {
                "stop_reason": "tool_use",
                "tool_calls": [
                    {
                        "id": "t2",
                        "name": "policy_check",
                        "input": {"order_id": "o-900", "reason": "damaged", "days_since_delivery": 3},
                    },
                    {"id": "t3", "name": "create_refund", "input": {"order_id": "o-900", "amount": 42.5}},
                ],
            },
            {"stop_reason": "end_turn", "output_text": "Refund submitted."},
        ]
    )
    result = run_agent("Refund my damaged item", model, TOOL_REGISTRY)
    assert len(_tool_messages(result["messages"])) == 2
    assert REFUNDS and REFUNDS[-1]["order_id"] == "o-900"

    # 3) Missing args -> is_error tool message
    model = ScriptedModel(
        [
            {"stop_reason": "tool_use", "tool_calls": [{"id": "bad-args", "name": "get_orders", "input": {}}]},
            {"stop_reason": "end_turn", "output_text": "Handled error."},
        ]
    )
    result = run_agent("debug", model, TOOL_REGISTRY)
    tool_msg = _tool_messages(result["messages"])[0]
    assert tool_msg["is_error"] is True
    assert "missing_required_args" in tool_msg["content"]

    # 4) Runtime exception -> is_error tool message
    model = ScriptedModel(
        [
            {"stop_reason": "tool_use", "tool_calls": [{"id": "boom", "name": "get_orders", "input": {"user_id": "boom"}}]},
            {"stop_reason": "end_turn", "output_text": "Handled exception."},
        ]
    )
    result = run_agent("debug", model, TOOL_REGISTRY)
    tool_msg = _tool_messages(result["messages"])[0]
    assert tool_msg["is_error"] is True
    assert "backend unavailable" in tool_msg["content"]

    # 5) Max step protection
    model = ScriptedModel(
        [
            {"stop_reason": "tool_use", "tool_calls": [{"id": "loop1", "name": "get_orders", "input": {"user_id": "u-200"}}]},
            {"stop_reason": "tool_use", "tool_calls": [{"id": "loop2", "name": "get_orders", "input": {"user_id": "u-200"}}]},
            {"stop_reason": "tool_use", "tool_calls": [{"id": "loop3", "name": "get_orders", "input": {"user_id": "u-200"}}]},
        ]
    )
    try:
        run_agent("loop", model, TOOL_REGISTRY, max_steps=2)
        raise AssertionError("Expected max_steps_exceeded")
    except RuntimeError as exc:
        assert "max_steps_exceeded" in str(exc)

    print("01_mock tests passed")


run_exam01_tests()
